In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
os.makedirs('../models', exist_ok=True)

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.12.1+cu130


In [17]:
DATA_PATH = '../data/'

normal_df = pd.read_csv(f'{DATA_PATH}normal3.csv', header=None, names=['raw', 'baseline', 'delta', 'current'])
slowed_df = pd.read_csv(f'{DATA_PATH}slowed3.csv', header=None, names=['raw', 'baseline', 'delta', 'current'])

print(f"Normal: {normal_df.shape}, Slowed: {slowed_df.shape}")

Normal: (1280, 4), Slowed: (1280, 4)


In [18]:
WINDOW_SIZE = 128

def create_windows(data, window_size):
    windows = []
    for i in range(len(data) - window_size + 1):
        windows.append(data[i:i+window_size])
    return np.array(windows)

X = np.vstack([
    create_windows(normal_df['current'].values, WINDOW_SIZE),
    create_windows(slowed_df['current'].values, WINDOW_SIZE)
])
y = np.hstack([
    np.zeros(len(normal_df) - WINDOW_SIZE + 1),
    np.ones(len(slowed_df) - WINDOW_SIZE + 1)
])

indices = np.random.permutation(len(X))
X, y = X[indices], y[indices]

print(f"Total samples: {X.shape}")

Total samples: (2306, 128)


In [19]:
scaler = MinMaxScaler()
X = scaler.fit_transform(X.reshape(-1, 1)).reshape(-1, WINDOW_SIZE)
joblib.dump(scaler, '../models/scaler.joblib')

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.125, random_state=42, stratify=y_temp)

print(f"Train: {X_train.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test.shape[0]}")

Train: 1613, Val: 231, Test: 462


In [20]:
class MotorFaultCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 32, 7, padding=3)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(2)
        self.dropout1 = nn.Dropout(0.15)
        
        self.conv2 = nn.Conv1d(32, 64, 5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(2)
        self.dropout2 = nn.Dropout(0.2)
        
        self.conv3 = nn.Conv1d(64, 128, 3, padding=1)
        self.bn3 = nn.BatchNorm1d(128)
        self.pool3 = nn.MaxPool1d(2)
        self.dropout3 = nn.Dropout(0.3)
        
        self.fc1 = nn.Linear(128 * 16, 64)
        self.bn_fc1 = nn.BatchNorm1d(64)
        self.dropout_fc = nn.Dropout(0.4)
        self.fc2 = nn.Linear(64, 2)
        
    def forward(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.dropout1(x)
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        x = self.dropout2(x)
        x = self.pool3(F.relu(self.bn3(self.conv3(x))))
        x = self.dropout3(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.bn_fc1(self.fc1(x)))
        x = self.dropout_fc(x)
        x = self.fc2(x)
        return x

model = MotorFaultCNN()
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Total parameters: 167,106


In [21]:
BATCH_SIZE = 32

train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train, dtype=torch.float32).unsqueeze(1),
        torch.tensor(y_train, dtype=torch.long)
    ),
    batch_size=BATCH_SIZE, shuffle=True
)
val_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_val, dtype=torch.float32).unsqueeze(1),
        torch.tensor(y_val, dtype=torch.long)
    ),
    batch_size=BATCH_SIZE, shuffle=False
)
test_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_test, dtype=torch.float32).unsqueeze(1),
        torch.tensor(y_test, dtype=torch.long)
    ),
    batch_size=BATCH_SIZE, shuffle=False
)

In [22]:
def train_model(model, train_loader, val_loader, epochs=50, lr=0.001):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
    
    best_val_acc = 0
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            _, pred = torch.max(outputs, 1)
            train_total += batch_y.size(0)
            train_correct += (pred == batch_y).sum().item()
        
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for batch_X, batch_y in val_loader:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()
                _, pred = torch.max(outputs, 1)
                val_total += batch_y.size(0)
                val_correct += (pred == batch_y).sum().item()
        
        val_acc = 100 * val_correct / val_total
        scheduler.step(val_loss)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), '../models/best_model.pth')
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= 15:
            break
    
    model.load_state_dict(torch.load('../models/best_model.pth'))
    return model, best_val_acc

model, best_val_acc = train_model(model, train_loader, val_loader)
print(f"Best validation accuracy: {best_val_acc:.2f}%")

Best validation accuracy: 100.00%


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.numpy())
        all_probs.extend(probs.cpu().numpy()[:, 1])

accuracy = 100 * np.mean(np.array(all_preds) == np.array(all_labels))
print(f"Test accuracy: {accuracy:.2f}%")

Test accuracy: 100.00%


In [23]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.numpy())

accuracy = 100 * np.mean(np.array(all_preds) == np.array(all_labels))
print(f"PyTorch Test Accuracy: {accuracy:.2f}%")

PyTorch Test Accuracy: 100.00%


In [24]:
import tensorflow as tf
import onnx2tf
import shutil
import os
import numpy as np

# Clean previous exports
for path in ["../models/model_tf", "../models/model_tf_quant"]:
    if os.path.exists(path):
        shutil.rmtree(path)

# Export to ONNX with explicit shape (1, 1, 128)
dummy_input = torch.randn(1, 1, 128)
onnx_path = "../models/model.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=11,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},   # Only batch dimension dynamic
        'output': {0: 'batch_size'}
    }
)
print("✅ Exported to ONNX")

# Convert to TensorFlow with explicit input shape
# The model expects (batch, channels, time) = (None, 1, 128)
# We need to tell onnx2tf to handle this correctly
onnx2tf.convert(
    input_onnx_file_path=onnx_path,
    output_folder_path="../models/model_tf",
    output_signaturedefs=True,
    # Force the input shape to be (None, 128, 1) for TFLite
    # onnx2tf will transpose automatically
)

print("✅ Converted to TensorFlow")

# Verify the SavedModel input shape
import tensorflow as tf
saved_model = tf.saved_model.load("../models/model_tf")
concrete_func = saved_model.signatures['serving_default']
print(f"SavedModel input shape: {concrete_func.inputs[0].shape}")
print(f"SavedModel input dtype: {concrete_func.inputs[0].dtype}")

# Test the SavedModel directly
test_input = np.random.randn(1, 128, 1).astype(np.float32)
output = concrete_func(tf.constant(test_input))
print(f"SavedModel test output: {output}")

# Now convert to TFLite
converter = tf.lite.TFLiteConverter.from_saved_model("../models/model_tf")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()
with open("../models/model_fp16.tflite", "wb") as f:
    f.write(tflite_model)
print("✅ FP16 TFLite model saved")

# FP32 version
converter = tf.lite.TFLiteConverter.from_saved_model("../models/model_tf")
tflite_model = converter.convert()
with open("../models/model_fp32.tflite", "wb") as f:
    f.write(tflite_model)
print("✅ FP32 TFLite model saved")

W0712 07:44:17.425000 186726 site-packages/torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 11 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MotorFaultCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MotorFaultCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Traceback (most recent call last):
  File "/home/jonny/miniconda3/envs/qai_hub/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
  File "/home/jonny/miniconda3/envs/qai_hub/lib/python3.10/site-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "/home/jonny/miniconda3/envs/qai_hub/lib/python3.10/site-packages/onnxscript/version_converter/__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
  File "/home/jonny/miniconda3/envs/qai_hub/lib/python3.10/site-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
RuntimeError: /github/workspace/onnx/version_converter/BaseConverter.h:68: adapter_lookup: Assertion `false` failed: No Adapter From Version $16 for Identity


[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✅ Exported to ONNX

Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Constant   │ 11             │ 11               │
│ Conv       │ 3              │ 3                │
│ Gemm       │ 2              │ 2                │
│ MaxPool    │ 3              │ 3                │
│ Relu       │ 4              │ 4                │
│ Reshape    │ 1              │ 1                │
│ Model Size │ 664.9KiB       │ 652.7KiB         │
└────────────┴────────────────┴──────────────────┘

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ C

W0000 00:00:1783822459.078968  186726 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1783822459.078982  186726 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-07-12 07:44:19.079084: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp1c_24fu5
2026-07-12 07:44:19.079563: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-07-12 07:44:19.079568: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmp1c_24fu5
2026-07-12 07:44:19.081462: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-07-12 07:44:19.085158: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmp1c_24fu5
2026-07-12 07:44:19.088366: I tensorflow/cc/saved_model/loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 9281 microseconds.
2026-07-12 07:44:19.118634: I tensorflow/compi

...

W0000 00:00:1783822459.628182  186726 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1783822459.628196  186726 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-07-12 07:44:19.628296: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpy5omqv91
2026-07-12 07:44:19.628733: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-07-12 07:44:19.628737: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpy5omqv91
2026-07-12 07:44:19.630427: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-07-12 07:44:19.634036: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpy5omqv91
2026-07-12 07:44:19.636735: I tensorflow/cc/saved_model/loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 8440 microseconds.
2026-07-12 07:44:19.666651: I tensorflow/compi

Float16 tflite output complete!
✅ Converted to TensorFlow


W0000 00:00:1783822460.085684  186726 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1783822460.085705  186726 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-07-12 07:44:20.085853: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: ../models/model_tf
2026-07-12 07:44:20.086224: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-07-12 07:44:20.086230: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: ../models/model_tf
2026-07-12 07:44:20.087724: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-07-12 07:44:20.091456: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: ../models/model_tf
2026-07-12 07:44:20.095847: I tensorflow/cc/saved_model/loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 9997 microseconds.
2026-07-12 07:44:20.124976: I tensorflow

✅ FP16 TFLite model saved to ../models/model_fp16.tflite
✅ FP32 TFLite model saved to ../models/model_fp32.tflite


W0000 00:00:1783822460.512976  186726 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1783822460.512995  186726 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.
2026-07-12 07:44:20.513153: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: ../models/model_tf
2026-07-12 07:44:20.513497: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-07-12 07:44:20.513503: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: ../models/model_tf
2026-07-12 07:44:20.514971: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-07-12 07:44:20.518583: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: ../models/model_tf
2026-07-12 07:44:20.521643: I tensorflow/cc/saved_model/loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 8493 microseconds.
2026-07-12 07:44:20.548626: I tensorflow

In [25]:
def verify_tflite(path):
    interpreter = tf.lite.Interpreter(model_path=path)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()
    out = interpreter.get_output_details()
    print(f"  Input: {inp[0]['shape']}, Output: {out[0]['shape']}")
    return interpreter

print("FP32 Model:")
verify_tflite("../models/model_fp32.tflite")

print("\nFP16 Model:")
verify_tflite("../models/model_fp16.tflite")

FP32 Model:
  Input: [  1 128   1], Output: [1 2]

FP16 Model:
  Input: [  1 128   1], Output: [1 2]


In [28]:
# Compare PyTorch and TFLite predictions on same samples
def compare_pytorch_tflite(sample_window, tflite_path):
    """Compare PyTorch and TFLite model outputs"""
    
    # PyTorch inference
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = MotorFaultCNN()
    model.load_state_dict(torch.load('../models/best_model.pth', map_location=device))
    model.to(device)
    model.eval()
    
    scaler = joblib.load('../models/scaler.joblib')
    
    # Preprocess for PyTorch - (batch, channels, time)
    window_scaled = scaler.transform(sample_window.reshape(-1, 1))
    input_torch = torch.tensor(window_scaled.reshape(1, 1, 128), dtype=torch.float32).to(device)
    
    with torch.no_grad():
        torch_output = model(input_torch).cpu().numpy()[0]
        torch_pred = np.argmax(torch_output)
        torch_conf = np.exp(torch_output[torch_pred]) / np.sum(np.exp(torch_output))
    
    # TFLite inference
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    # Preprocess for TFLite - (batch, time, channels)
    tflite_input = window_scaled.reshape(1, 128, 1).astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], tflite_input)
    interpreter.invoke()
    tflite_output = interpreter.get_tensor(output_details[0]['index'])[0]
    tflite_pred = np.argmax(tflite_output)
    tflite_conf = np.exp(tflite_output[tflite_pred]) / np.sum(np.exp(tflite_output))
    
    return {
        'torch': {'output': torch_output, 'pred': torch_pred, 'conf': float(torch_conf)},
        'tflite': {'output': tflite_output, 'pred': tflite_pred, 'conf': float(tflite_conf)}
    }

print("="*60)
print("PYTORCH vs FP32 TFLITE COMPARISON")
print("="*60)

normal_sample = normal_df['current'].values[:128]
fault_sample = slowed_df['current'].values[:128]

# Test FP32
result = compare_pytorch_tflite(normal_sample, "../models/model_fp32.tflite")
print(f"\nNormal Sample (FP32):")
print(f"  PyTorch:  {result['torch']['output']} -> {result['torch']['pred']} (conf: {result['torch']['conf']:.4f})")
print(f"  TFLite:   {result['tflite']['output']} -> {result['tflite']['pred']} (conf: {result['tflite']['conf']:.4f})")

result = compare_pytorch_tflite(fault_sample, "../models/model_fp32.tflite")
print(f"\nFault Sample (FP32):")
print(f"  PyTorch:  {result['torch']['output']} -> {result['torch']['pred']} (conf: {result['torch']['conf']:.4f})")
print(f"  TFLite:   {result['tflite']['output']} -> {result['tflite']['pred']} (conf: {result['tflite']['conf']:.4f})")

# Test FP16
result = compare_pytorch_tflite(normal_sample, "../models/model_fp16.tflite")
print(f"\nNormal Sample (FP16):")
print(f"  PyTorch:  {result['torch']['output']} -> {result['torch']['pred']} (conf: {result['torch']['conf']:.4f})")
print(f"  TFLite:   {result['tflite']['output']} -> {result['tflite']['pred']} (conf: {result['tflite']['conf']:.4f})")

result = compare_pytorch_tflite(fault_sample, "../models/model_fp16.tflite")
print(f"\nFault Sample (FP16):")
print(f"  PyTorch:  {result['torch']['output']} -> {result['torch']['pred']} (conf: {result['torch']['conf']:.4f})")
print(f"  TFLite:   {result['tflite']['output']} -> {result['tflite']['pred']} (conf: {result['tflite']['conf']:.4f})")

PYTORCH vs FP32 TFLITE COMPARISON

Normal Sample (FP32):
  PyTorch:  [ 2.0066445 -2.2789621] -> 0 (conf: 0.9864)
  TFLite:   [ 2.0066445 -2.2789629] -> 0 (conf: 0.9864)

Fault Sample (FP32):
  PyTorch:  [-0.62261367  0.43971288] -> 1 (conf: 0.7431)
  TFLite:   [-0.6226134   0.43971285] -> 1 (conf: 0.7431)

Normal Sample (FP16):
  PyTorch:  [ 2.0066445 -2.2789621] -> 0 (conf: 0.9864)
  TFLite:   [ 2.0066695 -2.2789252] -> 0 (conf: 0.9864)

Fault Sample (FP16):
  PyTorch:  [-0.62261367  0.43971288] -> 1 (conf: 0.7431)
  TFLite:   [-0.6227362   0.43975112] -> 1 (conf: 0.7432)


In [29]:
def evaluate_tflite(tflite_path, X_test, y_test):
    scaler = joblib.load('../models/scaler.joblib')
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    inp = interpreter.get_input_details()
    out = interpreter.get_output_details()
    
    predictions = []
    for window in X_test:
        window_scaled = scaler.transform(window.reshape(-1, 1))
        input_data = window_scaled.reshape(1, 128, 1).astype(np.float32)
        interpreter.set_tensor(inp[0]['index'], input_data)
        interpreter.invoke()
        output = interpreter.get_tensor(out[0]['index'])
        predictions.append(np.argmax(output[0]))
    
    return 100 * np.mean(np.array(predictions) == y_test)

print("="*60)
print("TFLITE TEST SET EVALUATION")
print("="*60)

acc_fp32 = evaluate_tflite("../models/model_fp32.tflite", X_test, y_test)
print(f"FP32 TFLite Test Accuracy: {acc_fp32:.2f}%")

acc_fp16 = evaluate_tflite("../models/model_fp16.tflite", X_test, y_test)
print(f"FP16 TFLite Test Accuracy: {acc_fp16:.2f}%")

# Optional: Compare file sizes
import os
fp32_size = os.path.getsize("../models/model_fp32.tflite") / 1024
fp16_size = os.path.getsize("../models/model_fp16.tflite") / 1024
print(f"\nFile sizes:")
print(f"  FP32: {fp32_size:.1f} KB")
print(f"  FP16: {fp16_size:.1f} KB")

TFLITE TEST SET EVALUATION
FP32 TFLite Test Accuracy: 50.00%
FP16 TFLite Test Accuracy: 50.00%

File sizes:
  FP32: 654.8 KB
  FP16: 330.7 KB
